In [1]:
!pip install faiss-cpu transformers sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.2 MB/s eta 0:00:00


In [2]:
import os, json, zipfile
import numpy as np
import pandas as pd
import faiss
import torch
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Extract Phase 1 outputs
zip_path = "/kaggle/input/notebooks/rajansamar/yolo-clip/_output_.zip"
extract_path = "/kaggle/working/phase1_outputs"

if not Path(extract_path).exists():
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("Extracted Phase 1 outputs")

PHASE1    = Path(extract_path)
CROPS_DIR = PHASE1 / "crops"
MANIFEST  = PHASE1 / "crops_manifest.csv"
DESC_FILE = Path("/kaggle/input/datasets/rajansamar/final-project/VR/Anno/list_description_inshop.json")
WORK_DIR  = Path("/kaggle/working/phase2")
WORK_DIR.mkdir(exist_ok=True)
print("Paths OK")

Device: cuda
Extracted Phase 1 outputs
Paths OK


In [3]:
with open(DESC_FILE) as f:
    desc_list = json.load(f)

caption_map = {}
for entry in desc_list:
    item_id = entry["item"]
    color   = entry.get("color", "")
    desc    = entry.get("description", [])
    text    = f"{color}. " + " ".join(desc[:2])
    caption_map[item_id] = text[:200]

print(f"Captions loaded: {len(caption_map)}")
print(f"Sample: {list(caption_map.items())[0]}")

Captions loaded: 7982
Sample: ('id_00000001', 'Cream. This sheer Georgette top features a high collar and shirred shoulders. Complete with long sleeves and buttoned cuffs.  Unlined')


In [4]:
df = pd.read_csv(MANIFEST)

# Fix crop paths to point to extracted location
df["crop_path"] = df["crop_path"].str.replace(
    "/kaggle/working/crops/",
    "/kaggle/working/phase1_outputs/crops/",
    regex=False
)

print(df["split"].value_counts())

gallery_df = df[df["split"] == "gallery"].reset_index(drop=True)
query_df   = df[df["split"] == "query"].reset_index(drop=True)

print(f"Gallery: {len(gallery_df)}, Query: {len(query_df)}")

def get_caption(item_id):
    return caption_map.get(item_id, "a clothing item")

gallery_df["caption"] = gallery_df["item_id"].apply(get_caption)
query_df["caption"]   = query_df["item_id"].apply(get_caption)

covered = gallery_df["item_id"].isin(caption_map).sum()
print(f"Gallery items with captions: {covered}/{len(gallery_df)}")

# Quick sanity check - verify first path exists
import os
print(f"First path exists: {os.path.exists(gallery_df['crop_path'].iloc[0])}")

split
train      25882
query      14218
gallery    12612
Name: count, dtype: int64
Gallery: 12612, Query: 14218
Gallery items with captions: 12612/12612
First path exists: True


In [5]:
MODEL_NAME = "openai/clip-vit-base-patch32"

clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model.eval()

print("CLIP loaded")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP loaded


In [6]:
BATCH_SIZE = 128

def embed_images(paths):
    all_embs = []
    for i in tqdm(range(0, len(paths), BATCH_SIZE), desc="Image embeds"):
        batch_paths = paths[i:i+BATCH_SIZE]
        images = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs = clip_processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)
        with torch.no_grad():
            vision_outputs = clip_model.vision_model(pixel_values=pixel_values)
            embs = clip_model.visual_projection(vision_outputs.pooler_output)
        embs = embs / embs.norm(dim=-1, keepdim=True)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs).astype("float32")

def embed_texts(texts):
    all_embs = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Text embeds"):
        batch = texts[i:i+BATCH_SIZE]
        inputs = clip_processor(
            text=batch, return_tensors="pt",
            padding=True, truncation=True, max_length=77
        )
        input_ids      = inputs["input_ids"].to(DEVICE)
        attention_mask = inputs["attention_mask"].to(DEVICE)
        with torch.no_grad():
            text_outputs = clip_model.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            embs = clip_model.text_projection(text_outputs.pooler_output)
        embs = embs / embs.norm(dim=-1, keepdim=True)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs).astype("float32")

def fuse(img_embs, txt_embs, alpha):
    fused = alpha * img_embs + (1 - alpha) * txt_embs
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    return (fused / norms).astype("float32")

print("Functions defined")

Functions defined


In [7]:
# Gallery
print("Embedding gallery images...")
gallery_img_embs = embed_images(gallery_df["crop_path"].tolist())

print("Embedding gallery captions...")
gallery_txt_embs = embed_texts(gallery_df["caption"].tolist())

# Query
print("Embedding query images...")
query_img_embs = embed_images(query_df["crop_path"].tolist())

print("Embedding query captions...")
query_txt_embs = embed_texts(query_df["caption"].tolist())

# Save all
np.save(WORK_DIR / "gallery_img_embs.npy", gallery_img_embs)
np.save(WORK_DIR / "gallery_txt_embs.npy", gallery_txt_embs)
np.save(WORK_DIR / "query_img_embs.npy",   query_img_embs)
np.save(WORK_DIR / "query_txt_embs.npy",   query_txt_embs)

gallery_df.to_csv(WORK_DIR / "gallery_meta.csv", index=False)
query_df.to_csv(WORK_DIR  / "query_meta.csv",   index=False)

print("All embeddings saved")

Embedding gallery images...


Image embeds: 100%|██████████| 99/99 [00:53<00:00,  1.86it/s]


Embedding gallery captions...


Text embeds: 100%|██████████| 99/99 [00:20<00:00,  4.90it/s]


Embedding query images...


Image embeds: 100%|██████████| 112/112 [01:04<00:00,  1.74it/s]


Embedding query captions...


Text embeds: 100%|██████████| 112/112 [00:23<00:00,  4.87it/s]


All embeddings saved


In [8]:
def build_hnsw_index(embeddings, M=32):
    dim   = embeddings.shape[1]
    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 128
    index.add(embeddings)
    return index

def search_index(index, query_embs, k=15):
    scores, indices = index.search(query_embs, k)
    return scores, indices

In [9]:
# === METRICS ===
def recall_at_k(query_ids, gallery_ids, retrieved_indices, k):
    hits = 0
    for i, qid in enumerate(query_ids):
        top_k = retrieved_indices[i][:k]
        if qid in [gallery_ids[j] for j in top_k]:
            hits += 1
    return hits / len(query_ids)

def ndcg_at_k(query_ids, gallery_ids, retrieved_indices, k):
    gallery_ids_arr = np.array(gallery_ids)
    scores = []
    for i, qid in enumerate(query_ids):
        top_k = retrieved_indices[i][:k]
        gains = [1 if gallery_ids[j] == qid else 0 for j in top_k]
        dcg   = sum(g / np.log2(r+2) for r, g in enumerate(gains))
        n_relevant = int(np.sum(gallery_ids_arr == qid))
        idcg  = sum(1 / np.log2(r+2) for r in range(min(n_relevant, k)))
        scores.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(scores)

def map_at_k(query_ids, gallery_ids, retrieved_indices, k):
    aps = []
    for i, qid in enumerate(query_ids):
        top_k = retrieved_indices[i][:k]
        hits, prec_sum = 0, 0
        for r, j in enumerate(top_k):
            if gallery_ids[j] == qid:
                hits += 1
                prec_sum += hits / (r+1)
        aps.append(prec_sum / max(hits, 1) if hits > 0 else 0)
    return np.mean(aps)

def evaluate(query_ids, gallery_ids, retrieved_indices, label=""):
    print(f"\n{'='*45}")
    print(f"  {label}")
    print(f"{'='*45}")
    results = []
    for k in [5, 10, 15]:
        r = recall_at_k(query_ids, gallery_ids, retrieved_indices, k)
        n = ndcg_at_k(query_ids, gallery_ids, retrieved_indices, k)
        m = map_at_k(query_ids, gallery_ids, retrieved_indices, k)
        print(f"  K={k:2d} | Recall={r:.4f} | NDCG={n:.4f} | mAP={m:.4f}")
        results.append({"config": label, "K": k, "Recall": r, "NDCG": n, "mAP": m})
    return results

# === ABLATION A ===
query_ids   = query_df["item_id"].tolist()
gallery_ids = gallery_df["item_id"].tolist()

index_A = build_hnsw_index(gallery_img_embs)
_, retrieved_A = search_index(index_A, query_img_embs, k=15)
results_A = evaluate(query_ids, gallery_ids, retrieved_A, label="A: Vision-only CLIP (α=1)")
faiss.write_index(index_A, str(WORK_DIR / "index_A.faiss"))


  A: Vision-only CLIP (α=1)
  K= 5 | Recall=0.4450 | NDCG=0.2055 | mAP=0.3277
  K=10 | Recall=0.5181 | NDCG=0.2091 | mAP=0.3229
  K=15 | Recall=0.5616 | NDCG=0.2159 | mAP=0.3156


In [10]:
all_results = results_A.copy()

for alpha in [0.7, 0.5]:
    gallery_fused = fuse(gallery_img_embs, gallery_txt_embs, alpha)
    query_fused   = fuse(query_img_embs,   query_txt_embs,   alpha)

    index_B = build_hnsw_index(gallery_fused)
    _, retrieved_B = search_index(index_B, query_fused, k=15)

    res = evaluate(query_ids, gallery_ids, retrieved_B,
                   label=f"B: Frozen CLIP + Captions (α={alpha})")
    all_results.extend(res)

    faiss.write_index(index_B, str(WORK_DIR / f"index_B_alpha{int(alpha*10)}.faiss"))


  B: Frozen CLIP + Captions (α=0.7)
  K= 5 | Recall=0.8516 | NDCG=0.6134 | mAP=0.7582
  K=10 | Recall=0.8785 | NDCG=0.6086 | mAP=0.7286
  K=15 | Recall=0.8920 | NDCG=0.6142 | mAP=0.7089

  B: Frozen CLIP + Captions (α=0.5)
  K= 5 | Recall=0.9897 | NDCG=0.9398 | mAP=0.9677
  K=10 | Recall=0.9943 | NDCG=0.9428 | mAP=0.9548
  K=15 | Recall=0.9960 | NDCG=0.9462 | mAP=0.9469


In [11]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(WORK_DIR / "ablation_results.csv", index=False)

print("\n=== FULL RESULTS TABLE ===")
print(results_df.to_string(index=False))
print(f"\nAll saved to {WORK_DIR}")


=== FULL RESULTS TABLE ===
                           config  K   Recall     NDCG      mAP
        A: Vision-only CLIP (α=1)  5 0.444999 0.205471 0.327719
        A: Vision-only CLIP (α=1) 10 0.518146 0.209147 0.322909
        A: Vision-only CLIP (α=1) 15 0.561612 0.215872 0.315606
B: Frozen CLIP + Captions (α=0.7)  5 0.851597 0.613370 0.758203
B: Frozen CLIP + Captions (α=0.7) 10 0.878464 0.608589 0.728577
B: Frozen CLIP + Captions (α=0.7) 15 0.891968 0.614198 0.708934
B: Frozen CLIP + Captions (α=0.5)  5 0.989661 0.939771 0.967748
B: Frozen CLIP + Captions (α=0.5) 10 0.994303 0.942811 0.954803
B: Frozen CLIP + Captions (α=0.5) 15 0.995991 0.946159 0.946936

All saved to /kaggle/working/phase2
